# Development measures and interventions

First inspect factor readability, fitted-subspace geometry, context consistency, and whole-history edit recovery across two smoke checkpoints. Then run four paired freeze branches from a saved checkpoint. The smoke results are pipeline examples, not factorization evidence. For an exploratory local run, use `runs/local/checkpoints/step_001000.pt` and train each arm for 1,000 more updates; branches inherit the parent's batch size, optimizer state, and scheduler state.

In [ ]:
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / 'pixi.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pixi.toml').exists(), 'Open this notebook from the project workspace'
import torch
torch.set_num_threads(1)
from factorization.analysis import analyze

RUN = ROOT / 'runs' / 'smoke'
DEV = RUN / 'analyses' / 'development-notebook-demo'
if not (DEV / 'results.json').exists():
    analyze(RUN, checkpoints=[0, 8], n_fit=32, n_test=32, pairs=8,
            interfaces=['input', 'L1_attn', 'L1_mlp', 'L4_mlp'], output=DEV)


In [ ]:
import json
results = json.loads((DEV / 'results.json').read_text())
for row in results['rows']:
    overlaps = [row['overlap'][i][j] for i in range(5) for j in range(i + 1, 5)
                if row['overlap'][i][j] is not None]
    print(f"step={row['step']} {row['interface']}: k95={row['pca']['k95']}, "
          f"natural R2={row['natural_encoding_r2']}, "
          f"mean factor cosine={sum(overlaps) / len(overlaps) if overlaps else None}")


In [ ]:
from IPython.display import display, Image
for name in ('development.png', 'edit_recovery.png', 'geometry.png'):
    path = DEV / 'figures' / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))


In [ ]:
from factorization.checkpoints import load_checkpoint
from factorization.train import train

CHECKPOINT = ROOT / 'runs' / 'smoke' / 'checkpoints' / 'step_000001.pt'
OUTPUT = ROOT / 'runs' / 'smoke' / 'branches' / 'notebook-demo'
parent = load_checkpoint(CHECKPOINT)
branch_config = dict(parent['config'], name='freeze_branch', steps=parent['step'] + 1)
for freeze in ('none', 'attention', 'mlp', 'embeddings'):
    branch_dir = OUTPUT / freeze
    if (branch_dir / 'config.json').exists():
        train(branch_config, branch_dir, device='cpu', resume=True)
    else:
        train(branch_config, branch_dir, device='cpu',
              parent_checkpoint=CHECKPOINT, freeze=freeze)


In [ ]:
# Inspect files written by the branch job.
for path in sorted(OUTPUT.rglob('*')):
    if path.is_file():
        print(path)